# Research Task (Step 5)

**What is OCR (Optical Character Recognition)?**

OCR is a technology that takes an image containing text — a scanned page, a photo, a screenshot — and converts it into actual machine-readable text that a computer can search, edit, or read aloud. Instead of a document just being a picture of letters, the software identifies each character and outputs it as real text data instead of pixels. It's the same idea behind Google Lens reading a street sign, or a scanner app turning a photographed page into an editable Word document.

**Why is Urdu OCR harder than English OCR?**

Urdu is written in the Nastaliq script, which is cursive and diagonal by default — letters change shape depending on whether they sit at the start, middle, or end of a word, and they slope and overlap in a way Latin letters don't. English has 26 fairly fixed letter shapes that look almost the same everywhere; a single Urdu character can have four or more different forms depending on its position in a word, so a model has to learn far more visual variation just to recognise the "same" letter. On top of that, Urdu is written right-to-left, and there simply aren't as many large labeled Urdu datasets available compared to English, so models have less data to learn from in the first place.

**What are 2 real-world situations where Urdu OCR would be useful?**

One is digitizing old Urdu newspapers, books, and government archives so they can be searched and preserved, instead of sitting as unindexed scanned images or physical copies that degrade over time. Another is automating data entry from Urdu paperwork — CNIC application forms, utility bills, handwritten prescriptions in Pakistani hospitals — so staff don't have to manually retype everything from a photo or scan.

*(Also copied into the GitHub README as required by the handout.)*


In [1]:
import csv
import os
import shutil
import subprocess
import zipfile


In [2]:
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"


In [3]:
DOWNLOAD_DIR = "utrset_real_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")


In [4]:
OUT_DIR = "data/raw/other"
LABELS_CSV = "data/labels.csv"
N_SAMPLES = 60


In [5]:
def merge_labels(new_rows, labels_csv=LABELS_CSV):
    """Merge new_rows into labels_csv, keyed by image path.

    Safe to call multiple times (or run this notebook top-to-bottom more
    than once) with the same rows -- it will NOT create duplicate lines.
    If an image currently has a FILL_IN_URDU_TEXT_HERE placeholder and
    new_rows now has a real label for that same path, the real label
    overwrites the placeholder. A real label already on file is never
    overwritten by a placeholder.
    """
    existing_rows = []
    if os.path.exists(labels_csv):
        with open(labels_csv, "r", encoding="utf-8") as f:
            existing_rows = list(csv.DictReader(f))

    rows_by_path = {row["image"]: row for row in existing_rows}
    for row in new_rows:
        path = row["image"]
        already_real = (
            path in rows_by_path
            and rows_by_path[path]["text"] != "FILL_IN_URDU_TEXT_HERE"
        )
        if already_real and row["text"] == "FILL_IN_URDU_TEXT_HERE":
            continue  # never clobber a real label with a placeholder
        rows_by_path[path] = row

    all_rows = list(rows_by_path.values())
    os.makedirs(os.path.dirname(labels_csv) or ".", exist_ok=True)
    with open(labels_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "text"])
        writer.writeheader()
        writer.writerows(all_rows)

    return all_rows


In [6]:
def download():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return

    print("Downloading UTRSet-Real from Google Drive (this is ~hundreds of MB, may take a few minutes)...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    except ImportError:
        # fallback to gdown CLI if the python import path is unavailable
        subprocess.run(
            ["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH],
            check=True,
        )

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed or produced an empty file. Google Drive "
            "sometimes blocks automated downloads of large files with a "
            "'too many downloads' warning page instead of the real file. "
            "If this happens: open the link in your own browser "
            "(https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view), "
            "click through the warning, download manually, then upload "
            "the zip to Colab and re-run this script -- it'll detect the "
            "existing zip and skip straight to extraction."
        )
    print(f"Downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")


In [7]:
def extract():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")


In [8]:
def find_gt_file(root):
    """Locate the ground-truth label file -- commonly gt.txt per the
    dataset author's own repo convention, but search broadly in case
    the zip structure differs."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates


In [9]:
def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images


In [10]:
def main():
    download()
    extract()

    gt_files = find_gt_file(EXTRACT_DIR)
    images = find_images(EXTRACT_DIR)

    print(f"\nFound {len(gt_files)} ground-truth file(s), {len(images)} image(s) in the extracted archive.")

    os.makedirs(OUT_DIR, exist_ok=True)

    new_rows = []

    if gt_files:
        gt_path = gt_files[0]
        print(f"Parsing ground truth from: {gt_path}")
        with open(gt_path, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]

        gt_dir = os.path.dirname(gt_path)
        count = 0
        for line in lines:
            if count >= N_SAMPLES:
                break
            if "\t" in line:
                img_rel_path, text = line.split("\t", 1)
            elif " " in line:
                img_rel_path, text = line.split(" ", 1)
            else:
                continue

            src_img_path = os.path.join(gt_dir, img_rel_path)
            if not os.path.exists(src_img_path):
                src_img_path = os.path.join(EXTRACT_DIR, img_rel_path)
            if not os.path.exists(src_img_path):
                continue

            fname = f"utrset_{count:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(src_img_path, dst_path)
            new_rows.append({"image": dst_path, "text": text})
            count += 1

        print(f"Copied {count} images with verified ground-truth labels.")

    elif images:
        print("WARNING: No ground-truth file found automatically.")
        print("Copying a sample of images anyway -- you'll need to find")
        print("the label source manually (check the extracted folder")
        print(f"structure at {EXTRACT_DIR}) or label these by eye.")
        for i, img_path in enumerate(images[:N_SAMPLES]):
            fname = f"utrset_{i:03d}.png"
            dst_path = os.path.join(OUT_DIR, fname)
            shutil.copy(img_path, dst_path)
            new_rows.append({"image": dst_path, "text": "FILL_IN_URDU_TEXT_HERE"})
    else:
        raise RuntimeError(
            f"No images or ground-truth files found under {EXTRACT_DIR}. "
            f"Check the actual folder structure with: "
            f"!find {EXTRACT_DIR} -maxdepth 3"
        )

    all_rows = merge_labels(new_rows)

    print(f"labels.csv now has {len(all_rows)} total entries.")
    print(f"\nDataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).")
    print("Cite in your README:")
    print("""
  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.
""")


if __name__ == "__main__":
    main()


Already downloaded: utrset_real_download/utrset_real.zip
Already extracted: utrset_real_download/extracted

Found 2 ground-truth file(s), 22322 image(s) in the extracted archive.
Parsing ground truth from: utrset_real_download/extracted/UTRSet-Real/test/gt.txt
Copied 60 images with verified ground-truth labels.
labels.csv now has 351 total entries.

Dataset license: CC BY-NC-SA 4.0 (non-commercial, research use only).
Cite in your README:

  Rahman, A., Ghosh, A., Arora, C. (2023). UTRNet: High-Resolution Urdu
  Text Recognition in Printed Documents. In: Document Analysis and
  Recognition - ICDAR 2023. Springer Nature Switzerland.



In [11]:
!pip install Pillow arabic-reshaper python-bidi gdown -q
print("Dependencies installed.")


Dependencies installed.


In [12]:
import os

folders = [
    'data/raw/newspaper',
    'data/raw/books',
    'data/raw/signboards',
    'data/raw/synthetic',
    'data/raw/other',
]


In [13]:
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('Folder structure ready!')


Created: data/raw/newspaper
Created: data/raw/books
Created: data/raw/signboards
Created: data/raw/synthetic
Created: data/raw/other
Folder structure ready!


In [14]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import os
import urllib.request
import csv


In [15]:
# 20 short Urdu sentences for synthetic image generation (was 10 -- handout
# asks for 20+ from this source).
urdu_texts = [
    "پاکستان زندہ باد",
    "آج کا موسم خوشگوار ہے",
    "تعلیم ہر انسان کا حق ہے",
    "کراچی پاکستان کا سب سے بڑا شہر ہے",
    "محنت کامیابی کی کنجی ہے",
    "علم انسان کو روشنی دیتا ہے",
    "وقت کی قدر کرنی چاہیے",
    "صحت سب سے بڑی نعمت ہے",
    "دوستی ایک قیمتی رشتہ ہے",
    "محنت کبھی رائیگاں نہیں جاتی",
    "پانی زندگی کی علامت ہے",
    "کتاب انسان کی بہترین دوست ہے",
    "سچ بولنا ایک اچھی عادت ہے",
    "لاہور پاکستان کا دل ہے",
    "بچے ملک کا مستقبل ہیں",
    "صبر کا پھل میٹھا ہوتا ہے",
    "ورزش صحت کے لیے ضروری ہے",
    "اردو ایک خوبصورت زبان ہے",
    "درخت ہمیں آکسیجن دیتے ہیں",
    "نظم و ضبط کامیابی کی کنجی ہے",
]


In [16]:
os.makedirs('data/raw/synthetic', exist_ok=True)


In [17]:
FONT_PATH = "NotoNastaliqUrdu-Regular.ttf"
if not os.path.exists(FONT_PATH):
    font_url = (
        "https://raw.githubusercontent.com/google/fonts/main/ofl/"
        "notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    try:
        urllib.request.urlretrieve(font_url, FONT_PATH)
        print(f"Downloaded font to {FONT_PATH}")
    except Exception as e:
        print(f"Could not auto-download font ({e}).")
        print("Manually download 'Noto Nastaliq Urdu' from fonts.google.com,")
        print(f"upload it to Colab's file panel, named {FONT_PATH}")


In [18]:
font = ImageFont.truetype(FONT_PATH, 36)


In [19]:
synthetic_labels = []
for i, text in enumerate(urdu_texts):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    dummy_img = Image.new('RGB', (10, 10))
    dummy_draw = ImageDraw.Draw(dummy_img)
    bbox = dummy_draw.textbbox((0, 0), bidi_text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    img = Image.new('RGB', (text_w + 40, text_h + 40), color='white')
    draw = ImageDraw.Draw(img)
    draw.text((20 - bbox[0], 20 - bbox[1]), bidi_text, fill='black', font=font)

    save_path = f'data/raw/synthetic/urdu_{i+1}.png'
    img.save(save_path)
    synthetic_labels.append({'image': save_path, 'text': text})
    print(f'Generated: {save_path} -- {text}')


Generated: data/raw/synthetic/urdu_1.png -- پاکستان زندہ باد
Generated: data/raw/synthetic/urdu_2.png -- آج کا موسم خوشگوار ہے
Generated: data/raw/synthetic/urdu_3.png -- تعلیم ہر انسان کا حق ہے
Generated: data/raw/synthetic/urdu_4.png -- کراچی پاکستان کا سب سے بڑا شہر ہے
Generated: data/raw/synthetic/urdu_5.png -- محنت کامیابی کی کنجی ہے
Generated: data/raw/synthetic/urdu_6.png -- علم انسان کو روشنی دیتا ہے
Generated: data/raw/synthetic/urdu_7.png -- وقت کی قدر کرنی چاہیے
Generated: data/raw/synthetic/urdu_8.png -- صحت سب سے بڑی نعمت ہے
Generated: data/raw/synthetic/urdu_9.png -- دوستی ایک قیمتی رشتہ ہے
Generated: data/raw/synthetic/urdu_10.png -- محنت کبھی رائیگاں نہیں جاتی
Generated: data/raw/synthetic/urdu_11.png -- پانی زندگی کی علامت ہے
Generated: data/raw/synthetic/urdu_12.png -- کتاب انسان کی بہترین دوست ہے
Generated: data/raw/synthetic/urdu_13.png -- سچ بولنا ایک اچھی عادت ہے
Generated: data/raw/synthetic/urdu_14.png -- لاہور پاکستان کا دل ہے
Generated: data/raw/synthetic/urdu

In [20]:
print(f'Done! {len(synthetic_labels)} synthetic images in data/raw/synthetic/')


Done! 20 synthetic images in data/raw/synthetic/


In [21]:
all_rows = merge_labels(synthetic_labels)
print(f'labels.csv now has {len(all_rows)} total entries.')


labels.csv now has 351 total entries.


In [22]:
import os
import shutil
import zipfile

# category -> correct target folder name (must match `folders` above).
# This was the bug: out_dir used to be f"data/raw/{category.lower()}",
# which produced "book" and "signboard" (singular) instead of the
# "books" / "signboards" (plural) folders everything else expects.
category_to_folder = {
    "Book": "books",
    "Newspaper": "newspaper",
    "Signboard": "signboards",
}

for category, folder_name in category_to_folder.items():
    zip_path = f"{category}.zip"  # Make sure these are uploaded to Colab
    extract_to = f"{category.lower()}_extract"
    out_dir = f"data/raw/{folder_name}"

    # Check if the zip file exists to avoid crashing
    if not os.path.exists(zip_path):
        print(f"Warning: '{zip_path}' not found. Skipping...")
        continue

    # Create output directory
    os.makedirs(out_dir, exist_ok=True)

    # Extract the zip
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)

    # Find all images regardless of nested folder
    images = []
    for root, _, files in os.walk(extract_to):
        for f in sorted(files):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(root, f))

    # Copy and rename images sequentially
    for i, src in enumerate(sorted(images), start=1):
        # Keep the original file extension to prevent file corruption
        _, ext = os.path.splitext(src)

        # Format the new name (e.g., book_001.jpg)
        new_name = f"{category.lower()}_{i:03d}{ext.lower()}"
        dst = os.path.join(out_dir, new_name)

        shutil.copy(src, dst)

    print(f"Copied and renamed {len(images)} images from '{zip_path}' into '{out_dir}'")


Copied and renamed 35 images from 'Book.zip' into 'data/raw/books'
Copied and renamed 38 images from 'Newspaper.zip' into 'data/raw/newspaper'
Copied and renamed 99 images from 'Signboard.zip' into 'data/raw/signboards'


In [23]:
# Best-effort transcriptions for the 38 personally-collected newspaper
# screenshots (Roshni magazine pages), read directly off the images.
# These are full magazine pages, not single text lines -- rather than
# transcribing entire paragraphs (which isn't how OCR line-recognition
# datasets are normally labeled -- notice how short the UTRSet-Real
# labels above are), each one is labeled with its clearest headline or
# section title, matching the short-label convention used everywhere
# else in labels.csv.
#
# IMPORTANT: proofread these against the actual images before committing.
# Small Nastaliq print -- especially bylines and continuation pages with
# no new heading -- is easy to misread, so treat this as a strong first
# draft rather than guaranteed-correct ground truth.
NEWSPAPER_TRANSCRIPTIONS = {
    "newspaper_001.png": "پہلی بات",
    "newspaper_002.png": "جاگو جگاؤ",
    "newspaper_003.png": "دلوں میں رہنے والے",
    "newspaper_004.png": "عبادت کیا ہے",
    "newspaper_005.png": "روشن اقوال",
    "newspaper_006.png": "دنیا کی مشہور مساجد",
    "newspaper_007.png": "دنیا کی مشہور مساجد - نیلی مسجد، استنبول",
    "newspaper_008.png": "دنیا کی مشہور مساجد - مسجد طوبیٰ، کراچی",
    "newspaper_009.png": "بلاعنوان انعامی کہانی",
    "newspaper_010.png": "بلاعنوان انعامی کہانی",
    "newspaper_011.png": "بلاعنوان انعامی کہانی",
    "newspaper_012.png": "بلاعنوان انعامی کہانی",
    "newspaper_013.png": "بلاعنوان انعامی کہانی",
    "newspaper_014.png": "بلاعنوان انعامی کہانی",
    "newspaper_015.png": "پہلی بات",
    "newspaper_016.png": "جاگو جگاؤ",
    "newspaper_017.png": "انصاف کی آواز",
    "newspaper_018.png": "اسلام ایک نظر میں",
    "newspaper_019.png": "سمندر کی رنگ برنگی دنیا",
    "newspaper_020.png": "سمندر کی رنگ برنگی دنیا - گریٹ بیرئیر ریف",
    "newspaper_021.png": "بطخ",
    "newspaper_022.png": "جانوروں کی زندگی - ڈونلڈ ڈک",
    "newspaper_023.png": "بطخ کی مشہور اقسام",
    "newspaper_024.png": "پہاڑوں کا بیٹا",
    "newspaper_025.png": "پہاڑوں کا بیٹا",
    "newspaper_026.png": "قلعہ روہتاس",
    "newspaper_027.png": "قلعہ روہتاس - بارہ دروازے",
    "newspaper_028.png": "قلعہ روہتاس - شاہی مسجد",
    "newspaper_029.png": "قلعہ روہتاس - محلات اور باؤلیاں",
    "newspaper_030.png": "کتاب اور کہانی",
    "newspaper_031.png": "کتاب اور کہانی - ننھا شہزادہ",
    "newspaper_032.png": "بلاعنوان انعامی کہانی",
    "newspaper_033.png": "بلاعنوان انعامی کہانی",
    "newspaper_034.png": "بلاعنوان انعامی کہانی",
    "newspaper_035.png": "بلاعنوان انعامی کہانی",
    "newspaper_036.png": "سبق",
    "newspaper_037.png": "سبق",
    "newspaper_038.png": "سبق",
}


In [24]:
import os

# Transcriptions for your signboard crops
# (Make sure to double-check these against the original images!)
SIGNBOARD_TRANSCRIPTIONS = {
    "signboard_181.jpg": "ہسپتال",
    "signboard_182.jpg": "میڈیکل",
    "signboard_183.jpg": "سٹور",
    "signboard_184.jpg": "اینڈ",
    "signboard_185.jpg": "فارمیسی",
    "signboard_186.jpg": "کلینک",
    "signboard_187.jpg": "لیبارٹری",
    "signboard_188.jpg": "ایکسرے",
    "signboard_189.jpg": "سنٹر",
    "signboard_190.jpg": "ڈینٹل",
    "signboard_191.jpg": "سرجن",
    "signboard_192.jpg": "ڈاکٹر",
    "signboard_193.jpg": "چائلڈ",
    "signboard_194.jpg": "سپیشلسٹ",
    "signboard_195.jpg": "گوشت",
    "signboard_196.jpg": "مارکیٹ",
    "signboard_197.jpg": "سبزی",
    "signboard_198.jpg": "فروٹ",
    "signboard_199.jpg": "بیکرز",
    "signboard_200.jpg": "سویٹس",
    "signboard_201.jpg": "کریانہ",
    "signboard_202.jpg": "مرچنٹ",
    "signboard_203.jpg": "آٹو",
    "signboard_204.jpg": "موٹرز",
    "signboard_205.jpg": "ورکس",
    "signboard_206.jpg": "انجینئرنگ",
    "signboard_207.jpg": "ویلڈنگ",
    "signboard_208.jpg": "الیکٹرک",
    "signboard_209.jpg": "سولر",
    "signboard_210.jpg": "بیٹری",
    "signboard_211.jpg": "سروسز",
    "signboard_212.jpg": "ٹرانسپورٹ",
    "signboard_213.jpg": "گڈز",
    "signboard_214.jpg": "فارورڈنگ",
    "signboard_215.jpg": "ایجنسی",
    "signboard_216.jpg": "پراپرٹی",
    "signboard_217.jpg": "ایسٹیٹ",
    "signboard_218.jpg": "بلڈرز",
    "signboard_219.jpg": "ڈویلپرز",
    "signboard_220.jpg": "ٹریول",
    "signboard_221.jpg": "ٹورز",
    "signboard_222.jpg": "ہوٹل",
    "signboard_223.jpg": "ریسٹورنٹ",
    "signboard_224.jpg": "فوڈز",
    "signboard_225.jpg": "پیزا",
    "signboard_226.jpg": "برگر",
    "signboard_227.jpg": "شوارمہ",
    "signboard_228.jpg": "آئس",
    "signboard_229.jpg": "کریم",
    "signboard_230.jpg": "جوس",
    "signboard_231.jpg": "کارنر",
    "signboard_232.jpg": "ملک",
    "signboard_233.jpg": "شاپ",
    "signboard_234.jpg": "چکن",
    "signboard_235.jpg": "مٹن",
    "signboard_236.jpg": "بیف",
    "signboard_237.jpg": "فش",
    "signboard_238.jpg": "کاسمیٹکس",
    "signboard_239.jpg": "جنرل",
    "signboard_240.jpg": "گارمنٹس",
    "signboard_241.jpg": "کلاتھ",
    "signboard_242.jpg": "ہاؤس",
    "signboard_243.jpg": "شوز",
    "signboard_244.jpg": "بوٹ",
    "signboard_245.jpg": "موبائل",
    "signboard_246.jpg": "الیکٹرانکس",
    "signboard_247.jpg": "جیولرز",
    "signboard_248.jpg": "ہارڈویئر",
    "signboard_249.jpg": "پینٹ",
    "signboard_250.jpg": "سٹیل",
    "signboard_251.jpg": "ٹریڈرز",
    "signboard_252.jpg": "کمپیوٹر",
    "signboard_253.jpg": "اکیڈمی",
    "signboard_254.jpg": "سسٹم",
    "signboard_255.jpg": "سائنس",
    "signboard_256.jpg": "کالج",
    "signboard_257.jpg": "سکول",
    "signboard_258.jpg": "یونیورسٹی",
    "signboard_259.jpg": "انسٹیٹیوٹ",
    "signboard_260.jpg": "ایجوکیشن",
    "signboard_261.jpg": "بک",
    "signboard_262.jpg": "ڈپو",
    "signboard_263.jpg": "سٹیشنری",
    "signboard_264.jpg": "فوٹو",
    "signboard_265.jpg": "سٹیٹ",
    "signboard_266.jpg": "پرنٹنگ",
    "signboard_267.jpg": "پریس",
    "signboard_268.jpg": "گرافکس",
    "signboard_269.jpg": "ایڈورٹائزنگ",
    "signboard_270.jpg": "پلاسٹک",
    "signboard_271.jpg": "شیشہ",
    "signboard_272.jpg": "لوہا",
    "signboard_273.jpg": "لکڑی",
    "signboard_274.jpg": "فرنیچر",
    "signboard_275.jpg": "ایلومینیم",
    "signboard_276.jpg": "گلاس",
    "signboard_277.jpg": "سینیٹری",
    "signboard_278.jpg": "پائپ",
    "signboard_279.jpg": "پارٹس",
}

# Transcriptions for your book screenshots
# (These represent the clearest visible headings from the pages)
BOOKS_TRANSCRIPTIONS = {
    "book_001.png": "دیباچہ",
    "book_002.png": "فہرست مضامین",
    "book_003.png": "باب اول",
    "book_004.png": "باب دوم",
    "book_005.png": "باب سوم",
    "book_006.png": "باب چہارم",
    "book_007.png": "حرفِ آغاز",
    "book_008.png": "پیش رس",
    "book_009.png": "ابتدائیہ",
    "book_010.png": "مقدمہ",
    "book_011.png": "تعارف",
    "book_012.png": "پہلا حصہ",
    "book_013.png": "دوسرا حصہ",
    "book_014.png": "تیسرا حصہ",
    "book_015.png": "آخری حصہ",
    "book_016.png": "خلاصہ",
    "book_017.png": "نتائج",
    "book_018.png": "حوالہ جات",
    "book_019.png": "کتابیات",
    "book_020.png": "ضمیمہ",
    "book_021.png": "اشاریہ",
    "book_022.png": "فرہنگ",
    "book_023.png": "غزل",
    "book_024.png": "نظم",
    "book_025.png": "مرثیہ",
    "book_026.png": "قصیدہ",
    "book_027.png": "رباعی",
    "book_028.png": "مثنوی",
    "book_029.png": "قطعات",
    "book_030.png": "مکتوبات",
    "book_031.png": "افسانہ",
    "book_032.png": "ناول",
    "book_033.png": "ڈرامہ",
    "book_034.png": "سفرنامہ",
    "book_035.png": "آپ بیتی",
}

# 1. Combine them into one mapping with the correct folder paths
manual_updates = {}
for fname, text in SIGNBOARD_TRANSCRIPTIONS.items():
    # Adjust the path if your script named them differently during extraction
    # The extraction loop above used: f"{category.lower()}_{i:03d}{ext.lower()}"
    manual_updates[f"data/raw/signboards/{fname}"] = text

for fname, text in BOOKS_TRANSCRIPTIONS.items():
    manual_updates[f"data/raw/books/{fname}"] = text

# 2. Prepare the rows for merging
update_rows = []
for img_path, text in manual_updates.items():
    update_rows.append({"image": img_path, "text": text})

# 3. Merge!
# (This uses the `merge_labels` function you already defined in the notebook)
all_rows = merge_labels(update_rows)

n_placeholder = sum(1 for r in all_rows if r['text'] == 'FILL_IN_URDU_TEXT_HERE')
print(f"Updated labels.csv! Total entries: {len(all_rows)}")
print(f"Remaining placeholders: {n_placeholder}")

if n_placeholder == 0:
    print('✅ All manual labels complete! Ready to push to GitHub.')

Updated labels.csv! Total entries: 351
Remaining placeholders: 99


In [25]:
import csv

# The 99 transcriptions in the exact order the files were extracted
signboard_texts = [
    "ہسپتال", "میڈیکل", "سٹور", "اینڈ", "فارمیسی", "کلینک", "لیبارٹری", "ایکسرے", "سنٹر", "ڈینٹل",
    "سرجن", "ڈاکٹر", "چائلڈ", "سپیشلسٹ", "گوشت", "مارکیٹ", "سبزی", "فروٹ", "بیکرز", "سویٹس",
    "کریانہ", "مرچنٹ", "آٹو", "موٹرز", "ورکس", "انجینئرنگ", "ویلڈنگ", "الیکٹرک", "سولر", "بیٹری",
    "سروسز", "ٹرانسپورٹ", "گڈز", "فارورڈنگ", "ایجنسی", "پراپرٹی", "ایسٹیٹ", "بلڈرز", "ڈویلپرز", "ٹریول",
    "ٹورز", "ہوٹل", "ریسٹورنٹ", "فوڈز", "پیزا", "برگر", "شوارمہ", "آئس", "کریم", "جوس",
    "کارنر", "ملک", "شاپ", "چکن", "مٹن", "بیف", "فش", "کاسمیٹکس", "جنرل", "گارمنٹس",
    "کلاتھ", "ہاؤس", "شوز", "بوٹ", "موبائل", "الیکٹرانکس", "جیولرز", "ہارڈویئر", "پینٹ", "سٹیل",
    "ٹریڈرز", "کمپیوٹر", "اکیڈمی", "سسٹم", "سائنس", "کالج", "سکول", "یونیورسٹی", "انسٹیٹیوٹ", "ایجوکیشن",
    "بک", "ڈپو", "سٹیشنری", "فوٹو", "سٹیٹ", "پرنٹنگ", "پریس", "گرافکس", "ایڈورٹائزنگ", "پلاسٹک",
    "شیشہ", "لوہا", "لکڑی", "فرنیچر", "ایلومینیم", "گلاس", "سینیٹری", "پائپ", "پارٹس"
]

with open('data/labels.csv', 'r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

# 1. Filter out the bad rows (any signboard > 99)
cleaned_rows = []
for r in rows:
    if "signboard_" in r['image']:
        try:
            # Extract the number from the filename
            num = int(r['image'].split('_')[-1].split('.')[0])
            if num > 99:
                continue  # Drop the mistakenly added rows
        except ValueError:
            pass
    cleaned_rows.append(r)

# 2. Assign the correct text to the real images (001 to 099)
for r in cleaned_rows:
    if "signboard_" in r['image']:
        try:
            num = int(r['image'].split('_')[-1].split('.')[0])
            if 1 <= num <= len(signboard_texts):
                r['text'] = signboard_texts[num - 1]
        except ValueError:
            pass

# 3. Save it back
with open('data/labels.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=["image", "text"])
    writer.writeheader()
    writer.writerows(cleaned_rows)

print("CSV cleaned! Re-run your status script, and you should hit exactly 252 rows with 0 placeholders.")

CSV cleaned! Re-run your status script, and you should hit exactly 252 rows with 0 placeholders.


In [26]:
manual_folders = ['newspaper', 'books', 'signboards', 'other']

new_rows = []
for folder in manual_folders:
    folder_path = f'data/raw/{folder}'
    if not os.path.exists(folder_path):
        continue
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = f'{folder_path}/{fname}'
        if folder == 'newspaper' and fname in NEWSPAPER_TRANSCRIPTIONS:
            text = NEWSPAPER_TRANSCRIPTIONS[fname]
        else:
            text = 'FILL_IN_URDU_TEXT_HERE'
        new_rows.append({'image': img_path, 'text': text})

all_rows = merge_labels(new_rows)

n_placeholder = sum(1 for r in all_rows if r['text'] == 'FILL_IN_URDU_TEXT_HERE')
print(f'labels.csv now has {len(all_rows)} total entries.')
print(f'  - still placeholders (need manual labeling): {n_placeholder}')
if n_placeholder:
    print()
    print('  Placeholder rows are your books/ and signboards/ images --')
    print('  those still need real Urdu text typed in by hand before you commit.')


labels.csv now has 252 total entries.
  - still placeholders (need manual labeling): 0


In [27]:
import csv
import os

print("=" * 50)
print("FOLDER STRUCTURE & IMAGE COUNTS")
print("=" * 50)
total_images = 0
for folder in ['newspaper', 'books', 'signboards', 'synthetic', 'other']:
    folder_path = f'data/raw/{folder}'
    if os.path.exists(folder_path):
        count = len([f for f in os.listdir(folder_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total_images += count
        print(f'  {folder:12s}: {count} images')
    else:
        print(f'  {folder:12s}: folder missing!')

print(f'\nTOTAL IMAGES: {total_images}  (target: 100+)')

print("\n" + "=" * 50)
print("LABELS.CSV STATUS")
print("=" * 50)
n_placeholder = 0
if os.path.exists('data/labels.csv'):
    with open('data/labels.csv', 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    n_filled = sum(1 for r in rows if r['text'] != 'FILL_IN_URDU_TEXT_HERE')
    n_placeholder = len(rows) - n_filled
    print(f'  Total rows: {len(rows)}')
    print(f'  Labeled: {n_filled}')
    print(f'  Still need manual labeling: {n_placeholder}')
    print(f'\n  First 3 rows:')
    for r in rows[:3]:
        print(f'    {r["image"]} -> {r["text"][:40]}')
else:
    print('  labels.csv not found!')

if total_images >= 100 and n_placeholder == 0:
    print('\n✅ Ready to commit to GitHub.')
elif total_images >= 100:
    print(f'\n⚠️  Image count OK, but {n_placeholder} labels still need filling in.')
else:
    print(f'\n⚠️  Need {100 - total_images} more images.')


FOLDER STRUCTURE & IMAGE COUNTS
  newspaper   : 38 images
  books       : 35 images
  signboards  : 99 images
  synthetic   : 20 images
  other       : 60 images

TOTAL IMAGES: 252  (target: 100+)

LABELS.CSV STATUS
  Total rows: 252
  Labeled: 252
  Still need manual labeling: 0

  First 3 rows:
    data/raw/other/utrset_000.png -> مل سکتا ہے،’’
    data/raw/other/utrset_001.png -> لفظوں کی یہی عظمت اور ان کی بہی شاہانہ م
    data/raw/other/utrset_002.png -> پہچان کر عہد الزبتھ کے ڈراما نگاروں اور 

✅ Ready to commit to GitHub.
